# 🎯 Meetup Group Data Analysis

This notebook pulls event and RSVP data from the **1-5genasians** Meetup group via the GraphQL API and computes analytics on group health, host engagement, and member activity.

---

## 1. Setup & Dependencies

In [ ]:
# Install dependencies (run once)
%pip install requests pandas matplotlib seaborn python-dotenv --quiet

In [ ]:
import os
import json
import time
import webbrowser
from datetime import datetime
from urllib.parse import urlencode, urlparse, parse_qs

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv

# Load .env from project root
load_dotenv()

# Plotting defaults
sns.set_theme(style="darkgrid", palette="viridis")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

print("✅ Setup complete")

## 2. Authentication

We use the Meetup OAuth2 authorization-code flow.

1. Run the cell below — it opens the Meetup authorize URL in your browser.
2. Grant access. You'll be redirected (the redirect may fail to load — that's fine).
3. Copy the **full redirect URL** from your browser address bar and paste it when prompted.

Alternatively, if you already have a valid Bearer token, paste it directly into `ACCESS_TOKEN`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MEETUP_CLIENT_ID = os.getenv("MEETUP_KEY", "")
MEETUP_CLIENT_SECRET = os.getenv("MEETUP_SECRET", "")
REDIRECT_URI = "https://meetup-discord-bot.herokuapp.com/connect/meetup/callback"
GQL_ENDPOINT = "https://api.meetup.com/gql-ext"
GROUP_URLNAME = "1-5genasians"

# ── Option A: Paste a token you already have ─────────────────────────────────
ACCESS_TOKEN = ""  # <-- paste here if you have one

# ── Option B: OAuth2 flow ────────────────────────────────────────────────────
if not ACCESS_TOKEN:
    auth_url = (
        "https://secure.meetup.com/oauth2/authorize?"
        + urlencode({
            "client_id": MEETUP_CLIENT_ID,
            "response_type": "code",
            "redirect_uri": REDIRECT_URI,
            "scope": "basic",
        })
    )
    print("Opening browser for Meetup authorization...")
    print(f"If it doesn't open, visit:\n{auth_url}\n")
    webbrowser.open(auth_url)

    redirect_input = input("Paste the full redirect URL (or just the code): ").strip()

    # Extract code from full URL or bare code
    if redirect_input.startswith("http"):
        parsed = parse_qs(urlparse(redirect_input).query)
        code = parsed.get("code", [redirect_input])[0]
    else:
        code = redirect_input

    # Exchange code for token
    token_resp = requests.post(
        "https://secure.meetup.com/oauth2/access",
        data={
            "client_id": MEETUP_CLIENT_ID,
            "client_secret": MEETUP_CLIENT_SECRET,
            "grant_type": "authorization_code",
            "redirect_uri": REDIRECT_URI,
            "code": code,
        },
    )
    token_resp.raise_for_status()
    token_data = token_resp.json()
    ACCESS_TOKEN = token_data["access_token"]
    print(f"\n✅ Token acquired (expires in {token_data.get('expires_in', '?')}s)")

HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
print("🔑 Ready to query Meetup API")

## 3. Data Fetching Helpers

In [ ]:
def gql_request(query: str, variables: dict | None = None) -> dict:
    """Send a GraphQL request to the Meetup API."""
    resp = requests.post(
        GQL_ENDPOINT,
        headers=HEADERS,
        json={"query": query, "variables": variables or {}},
    )
    resp.raise_for_status()
    body = resp.json()
    if "errors" in body:
        raise RuntimeError(f"GraphQL errors: {json.dumps(body['errors'], indent=2)}")
    return body["data"]


def paginate(query: str, variables: dict, path_to_connection: list[str],
             page_size: int = 100, delay: float = 0.3) -> list[dict]:
    """
    Auto-paginate a Meetup GraphQL connection.
    `path_to_connection` is the dot-path from `data` to the connection object,
    e.g. ["groupByUrlname", "events"].
    """
    all_nodes = []
    cursor = None
    page = 0
    while True:
        vars_with_page = {**variables, "first": page_size}
        if cursor:
            vars_with_page["after"] = cursor

        data = gql_request(query, vars_with_page)

        # Navigate to the connection
        connection = data
        for key in path_to_connection:
            connection = connection[key]

        edges = connection.get("edges", [])
        all_nodes.extend(edge["node"] for edge in edges)

        page_info = connection.get("pageInfo", {})
        page += 1
        total = connection.get("totalCount", "?")
        print(f"  Page {page}: fetched {len(edges)} (total so far: {len(all_nodes)}/{total})")

        if not page_info.get("hasNextPage", False):
            break
        cursor = page_info["endCursor"]
        time.sleep(delay)  # be nice to the API

    return all_nodes

print("✅ Helpers loaded")

## 4. Fetch All Past Events

In [ ]:
GET_GROUP_EVENTS = """
query($urlname: String!, $first: Int!, $after: String, $filter: GroupEventFilter) {
  groupByUrlname(urlname: $urlname) {
    id
    events(first: $first, after: $after, filter: $filter) {
      pageInfo { hasNextPage endCursor }
      totalCount
      edges {
        node {
          id
          title
          dateTime
          eventUrl
          maxTickets
          status
          eventHosts {
            member { id name gender memberUrl }
          }
        }
      }
    }
  }
}
"""

print("Fetching all PAST events...")
raw_events = paginate(
    GET_GROUP_EVENTS,
    {"urlname": GROUP_URLNAME, "filter": {"status": ["PAST"]}},
    ["groupByUrlname", "events"],
)
print(f"\n✅ Fetched {len(raw_events)} past events")

In [ ]:
# Build events DataFrame
events_data = []
for ev in raw_events:
    hosts = ev.get("eventHosts") or []
    host_names = [h["member"]["name"] for h in hosts]
    host_ids = [h["member"]["id"] for h in hosts]
    host_genders = [h["member"]["gender"] for h in hosts]
    events_data.append({
        "event_id": ev["id"],
        "title": ev["title"],
        "datetime": pd.to_datetime(ev["dateTime"]),
        "url": ev["eventUrl"],
        "max_tickets": ev.get("maxTickets"),
        "status": ev["status"],
        "host_names": host_names,
        "host_ids": host_ids,
        "host_genders": host_genders,
        "num_hosts": len(hosts),
    })

df_events = pd.DataFrame(events_data)
df_events["year_month"] = df_events["datetime"].dt.to_period("M")
df_events["day_of_week"] = df_events["datetime"].dt.day_name()
df_events = df_events.sort_values("datetime").reset_index(drop=True)

print(f"Events DataFrame: {len(df_events)} rows")
df_events.head()

## 5. Fetch RSVPs for Every Event

This may take a while for large groups — we fetch YES, ATTENDED, and NO_SHOW RSVPs for each event.

In [ ]:
GET_EVENT_RSVPS = """
query($eventId: ID!, $first: Int!, $after: String) {
  event(id: $eventId) {
    id
    rsvps(first: $first, after: $after) {
      pageInfo { hasNextPage endCursor }
      totalCount
      edges {
        node {
          status
          member { id name gender memberUrl }
        }
      }
    }
  }
}
"""

all_rsvps = []
total = len(df_events)
for i, row in df_events.iterrows():
    eid = row["event_id"]
    print(f"[{i+1}/{total}] RSVPs for: {row['title'][:50]}...", end=" ")
    try:
        rsvps = paginate(
            GET_EVENT_RSVPS,
            {"eventId": eid},
            ["event", "rsvps"],
            delay=0.2,
        )
        for r in rsvps:
            member = r.get("member") or {}
            all_rsvps.append({
                "event_id": eid,
                "member_id": member.get("id"),
                "member_name": member.get("name"),
                "gender": member.get("gender"),
                "rsvp_status": r["status"],
            })
    except Exception as e:
        print(f"⚠️ Error: {e}")

df_rsvps = pd.DataFrame(all_rsvps)
print(f"\n✅ Fetched {len(df_rsvps)} total RSVPs across {total} events")
df_rsvps.head()

## 6. Data Enrichment

In [ ]:
# Merge RSVP counts back onto events
rsvp_counts = df_rsvps.groupby("event_id").agg(
    total_rsvps=("rsvp_status", "count"),
    yes_count=("rsvp_status", lambda x: (x == "YES").sum()),
    attended_count=("rsvp_status", lambda x: (x == "ATTENDED").sum()),
    no_show_count=("rsvp_status", lambda x: (x == "NO_SHOW").sum()),
    waitlist_count=("rsvp_status", lambda x: (x == "WAITLIST").sum()),
).reset_index()

df = df_events.merge(rsvp_counts, on="event_id", how="left")
df["total_rsvps"] = df["total_rsvps"].fillna(0).astype(int)

# Capacity utilization
df["capacity_util"] = df.apply(
    lambda r: r["total_rsvps"] / r["max_tickets"] if r["max_tickets"] and r["max_tickets"] > 0 else None,
    axis=1,
)

# Positive attendees = YES + ATTENDED (people who said yes or actually attended)
df["positive_rsvps"] = df["yes_count"].fillna(0) + df["attended_count"].fillna(0)

print(f"✅ Enriched DataFrame ready: {len(df)} events")
df[["title", "datetime", "total_rsvps", "positive_rsvps", "no_show_count", "capacity_util"]].head(10)

---
# 📊 Metrics & Visualizations

### Metric 1: Events Per Month

In [ ]:
events_per_month = df.groupby("year_month").size()

fig, ax = plt.subplots()
events_per_month.plot(kind="bar", ax=ax, color=sns.color_palette("viridis", len(events_per_month)))
ax.set_title("Events Per Month")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Events")
# Show every Nth label to avoid crowding
n = max(1, len(events_per_month) // 15)
for i, label in enumerate(ax.xaxis.get_ticklabels()):
    if i % n != 0:
        label.set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"Average: {events_per_month.mean():.1f} events/month")
print(f"Median:  {events_per_month.median():.1f} events/month")

### Metric 2: Average Events Per Month Per Active Member

An "active member" in a given month = someone who RSVP'd YES/ATTENDED to at least one event that month.

In [ ]:
# Merge event datetime onto RSVPs
rsvps_with_date = df_rsvps.merge(
    df_events[["event_id", "datetime"]],
    on="event_id",
)
rsvps_with_date["year_month"] = rsvps_with_date["datetime"].dt.to_period("M")

# Filter to positive RSVPs
positive = rsvps_with_date[rsvps_with_date["rsvp_status"].isin(["YES", "ATTENDED"])]

# Events attended per member per month
member_month = positive.groupby(["year_month", "member_id"]).agg(
    events_attended=("event_id", "nunique")
).reset_index()

# Average across members for each month
avg_events_per_active = member_month.groupby("year_month")["events_attended"].mean()

fig, ax = plt.subplots()
avg_events_per_active.plot(kind="line", ax=ax, marker="o", markersize=4, linewidth=1.5)
ax.set_title("Avg Events/Month Per Active Member")
ax.set_ylabel("Events Attended")
ax.set_xlabel("Month")
n = max(1, len(avg_events_per_active) // 15)
for i, label in enumerate(ax.xaxis.get_ticklabels()):
    if i % n != 0:
        label.set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"Overall average: {member_month['events_attended'].mean():.2f} events/month per active member")

### Metric 3: Host-to-Active-Member Ratio

In [ ]:
# Unique hosts (all-time)
all_host_ids = set()
for ids in df["host_ids"]:
    all_host_ids.update(ids)

# Unique active attendees (all-time): anyone who RSVP'd YES or ATTENDED at least once
active_member_ids = set(
    df_rsvps[df_rsvps["rsvp_status"].isin(["YES", "ATTENDED"])]["member_id"].dropna().unique()
)

num_hosts = len(all_host_ids)
num_active = len(active_member_ids)
ratio = num_hosts / num_active if num_active else 0

print(f"Unique hosts (all-time):          {num_hosts}")
print(f"Unique active members (all-time):  {num_active}")
print(f"Host-to-active-member ratio:       {ratio:.2%}  (1 host per {1/ratio:.0f} members)" if ratio else "N/A")

# Over time
host_ids_by_month = df.explode("host_ids").dropna(subset=["host_ids"]).groupby("year_month")["host_ids"].nunique()
active_by_month = positive.groupby("year_month")["member_id"].nunique()

ratio_by_month = (host_ids_by_month / active_by_month).dropna()

fig, ax = plt.subplots()
ratio_by_month.plot(kind="line", ax=ax, marker="o", markersize=4, linewidth=1.5, color="coral")
ax.set_title("Host-to-Active-Member Ratio Over Time")
ax.set_ylabel("Ratio (hosts / active members)")
ax.set_xlabel("Month")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
n = max(1, len(ratio_by_month) // 15)
for i, label in enumerate(ax.xaxis.get_ticklabels()):
    if i % n != 0:
        label.set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Metric 4: Top Hosts Leaderboard

In [ ]:
# Explode hosts so each host-event pair is a row
hosts_exploded = df.explode("host_names").explode("host_ids")

# De-duplicate (one row per host per event)
host_event_pairs = []
for _, row in df.iterrows():
    for hname, hid in zip(row["host_names"], row["host_ids"]):
        host_event_pairs.append({"host_name": hname, "host_id": hid, "event_id": row["event_id"]})

df_host_events = pd.DataFrame(host_event_pairs)
top_hosts = df_host_events.groupby(["host_id", "host_name"]).size().reset_index(name="events_hosted")
top_hosts = top_hosts.sort_values("events_hosted", ascending=False).reset_index(drop=True)

# Show top 20
top_n = top_hosts.head(20)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_n, y="host_name", x="events_hosted", ax=ax, palette="viridis")
ax.set_title("Top 20 Hosts by Events Hosted")
ax.set_xlabel("Events Hosted")
ax.set_ylabel("")
for i, v in enumerate(top_n["events_hosted"]):
    ax.text(v + 0.3, i, str(v), va="center", fontsize=10)
plt.tight_layout()
plt.show()

top_hosts.head(20)

### Metric 5: Average RSVP Count Per Event

In [ ]:
avg_rsvps_monthly = df.groupby("year_month")["positive_rsvps"].mean()

fig, ax = plt.subplots()
avg_rsvps_monthly.plot(kind="line", ax=ax, marker="o", markersize=4, linewidth=1.5, color="teal")
ax.set_title("Avg Positive RSVPs Per Event (by Month)")
ax.set_ylabel("Avg RSVPs (YES + ATTENDED)")
ax.set_xlabel("Month")
n = max(1, len(avg_rsvps_monthly) // 15)
for i, label in enumerate(ax.xaxis.get_ticklabels()):
    if i % n != 0:
        label.set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"Overall avg positive RSVPs per event: {df['positive_rsvps'].mean():.1f}")
print(f"Overall median:                       {df['positive_rsvps'].median():.1f}")

### Metric 6: RSVP Yes vs Attended vs No-Show Rate

In [ ]:
# Overall breakdown
status_counts = df_rsvps["rsvp_status"].value_counts()
print("Overall RSVP Status Breakdown:")
print(status_counts)
print()

# Pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall pie
status_counts.plot.pie(
    ax=axes[0], autopct="%1.1f%%", startangle=90,
    colors=sns.color_palette("viridis", len(status_counts)),
)
axes[0].set_title("RSVP Status Breakdown (All Time)")
axes[0].set_ylabel("")

# No-show rate over time
monthly_stats = df.groupby("year_month").agg(
    total_yes=pd.NamedAgg(column="yes_count", aggfunc="sum"),
    total_attended=pd.NamedAgg(column="attended_count", aggfunc="sum"),
    total_noshow=pd.NamedAgg(column="no_show_count", aggfunc="sum"),
)
monthly_stats["noshow_rate"] = monthly_stats["total_noshow"] / (
    monthly_stats["total_yes"] + monthly_stats["total_attended"] + monthly_stats["total_noshow"]
)

monthly_stats["noshow_rate"].plot(ax=axes[1], marker="o", markersize=4, linewidth=1.5, color="tomato")
axes[1].set_title("No-Show Rate Over Time")
axes[1].set_ylabel("No-Show Rate")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
n = max(1, len(monthly_stats) // 15)
for i, label in enumerate(axes[1].xaxis.get_ticklabels()):
    if i % n != 0:
        label.set_visible(False)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

overall_noshow = monthly_stats["total_noshow"].sum() / (
    monthly_stats["total_yes"].sum() + monthly_stats["total_attended"].sum() + monthly_stats["total_noshow"].sum()
)
print(f"Overall no-show rate: {overall_noshow:.1%}")

### Metric 7: Event Popularity by Day of Week

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Avg RSVPs by day of week
dow_stats = df.groupby("day_of_week").agg(
    num_events=("event_id", "count"),
    avg_rsvps=("positive_rsvps", "mean"),
    avg_capacity_util=("capacity_util", "mean"),
).reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Events count by day
sns.barplot(x=dow_stats.index, y=dow_stats["num_events"], ax=axes[0], palette="viridis")
axes[0].set_title("Number of Events by Day of Week")
axes[0].set_ylabel("Events")
axes[0].set_xlabel("")

# Avg RSVPs by day
sns.barplot(x=dow_stats.index, y=dow_stats["avg_rsvps"], ax=axes[1], palette="magma")
axes[1].set_title("Avg Positive RSVPs by Day of Week")
axes[1].set_ylabel("Avg RSVPs")
axes[1].set_xlabel("")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

dow_stats

### Metric 8: New vs Returning Attendees Over Time

In [ ]:
# Track first-seen month for each member
positive_sorted = positive.sort_values("datetime")
first_seen = positive_sorted.groupby("member_id")["year_month"].first().rename("first_month")

# Join back
pos_with_first = positive_sorted.merge(first_seen, on="member_id")
pos_with_first["is_new"] = pos_with_first["year_month"] == pos_with_first["first_month"]

new_vs_returning = pos_with_first.groupby(["year_month", "is_new"])["member_id"].nunique().unstack(fill_value=0)
new_vs_returning.columns = ["Returning", "New"]

fig, ax = plt.subplots(figsize=(14, 5))
new_vs_returning.plot(kind="bar", stacked=True, ax=ax, color=["#2ecc71", "#3498db"])
ax.set_title("New vs Returning Active Members Per Month")
ax.set_ylabel("Unique Members")
ax.set_xlabel("Month")
n = max(1, len(new_vs_returning) // 15)
for i, label in enumerate(ax.xaxis.get_ticklabels()):
    if i % n != 0:
        label.set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Member Type")
plt.tight_layout()
plt.show()

total_new = pos_with_first[pos_with_first["is_new"]]["member_id"].nunique()
total_returning = pos_with_first[~pos_with_first["is_new"]]["member_id"].nunique()
print(f"Total unique new members seen: {total_new}")
print(f"Members who returned after first month: {total_returning}")
print(f"Retention rate: {total_returning / total_new:.1%}" if total_new else "N/A")

### Metric 9: Gender Breakdown of Attendees

In [ ]:
# Unique attendees with gender info
attendees = df_rsvps[
    df_rsvps["rsvp_status"].isin(["YES", "ATTENDED"])
].drop_duplicates(subset="member_id")

gender_map = {
    "FEMALE": "Female",
    "MALE": "Male",
    "OTHER": "Other",
    "NONE": "Not Specified",
    "NOT_CHECKED": "Not Specified",
}
attendees = attendees.copy()
attendees["gender_label"] = attendees["gender"].map(gender_map).fillna("Not Specified")

gender_counts = attendees["gender_label"].value_counts()

fig, ax = plt.subplots(figsize=(7, 5))
gender_counts.plot.pie(
    ax=ax, autopct="%1.1f%%", startangle=90,
    colors=["#e74c3c", "#3498db", "#2ecc71", "#95a5a6"],
)
ax.set_title("Gender Breakdown of Active Attendees")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

print(gender_counts)

### Metric 10: Event Capacity Utilization

In [ ]:
# Filter to events that have a maxTickets value
df_with_cap = df[df["capacity_util"].notna()].copy()

if len(df_with_cap) > 0:
    avg_util_monthly = df_with_cap.groupby("year_month")["capacity_util"].mean()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribution
    sns.histplot(df_with_cap["capacity_util"], bins=20, ax=axes[0], color="teal", kde=True)
    axes[0].set_title("Distribution of Capacity Utilization")
    axes[0].set_xlabel("Utilization (RSVPs / Max Tickets)")
    axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

    # Over time
    avg_util_monthly.plot(ax=axes[1], marker="o", markersize=4, linewidth=1.5, color="darkorange")
    axes[1].set_title("Avg Capacity Utilization Over Time")
    axes[1].set_ylabel("Utilization")
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    n = max(1, len(avg_util_monthly) // 15)
    for i, label in enumerate(axes[1].xaxis.get_ticklabels()):
        if i % n != 0:
            label.set_visible(False)
    plt.xticks(rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

    print(f"Events with capacity set: {len(df_with_cap)} / {len(df)}")
    print(f"Avg utilization: {df_with_cap['capacity_util'].mean():.1%}")
    print(f"Median utilization: {df_with_cap['capacity_util'].median():.1%}")
else:
    print("No events have maxTickets set — skipping capacity utilization.")

---
## 📋 Summary Dashboard

In [ ]:
date_range = f"{df['datetime'].min().strftime('%b %Y')} – {df['datetime'].max().strftime('%b %Y')}"
months_active = df["year_month"].nunique()

summary = {
    "Date Range": date_range,
    "Total Events": len(df),
    "Months Active": months_active,
    "Avg Events/Month": f"{len(df) / months_active:.1f}",
    "Unique Hosts": num_hosts,
    "Unique Active Members": num_active,
    "Host:Member Ratio": f"1:{1/ratio:.0f}" if ratio else "N/A",
    "Avg RSVPs/Event": f"{df['positive_rsvps'].mean():.1f}",
    "No-Show Rate": f"{overall_noshow:.1%}",
    "Member Retention Rate": f"{total_returning / total_new:.1%}" if total_new else "N/A",
}

print("\n" + "="*50)
print("  📊 GROUP HEALTH SUMMARY")
print("="*50)
for k, v in summary.items():
    print(f"  {k:.<30s} {v}")
print("="*50)